# NumPy Basics -- Prerequisite Refresher

**Who is this for?** You already know NumPy. You've used arrays, sliced them, maybe cursed at broadcasting errors. This notebook is a quick refresher and self-check before the ML course begins.

**Estimated time:** 30--45 minutes.

**What it covers:**
1. Array creation
2. Indexing and slicing
3. Broadcasting
4. Vectorized operations
5. Reshaping
6. Why vectorization matters (timing)

If any section feels unfamiliar (not just rusty), check the references at the bottom and review before the course starts.

In [ ]:
import numpy as np
np.__version__

---
## 1. Array Creation

The core constructors you should know cold: `np.array`, `np.zeros`, `np.ones`, `np.arange`, `np.linspace`, and the `np.random` module.

In [ ]:
# From a list
a = np.array([1, 2, 3, 4])
print("array:", a, "| dtype:", a.dtype, "| shape:", a.shape)

# Common constructors
print("zeros:", np.zeros((2, 3)))
print("ones:", np.ones(5, dtype=np.int32))
print("arange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5))

In [ ]:
# Random arrays
rng = np.random.default_rng(42)  # modern API -- prefer this over np.random.seed
print("uniform:", rng.random((2, 3)))
print("normal:", rng.standard_normal((2, 3)))
print("integers:", rng.integers(0, 10, size=(3,)))

### Exercise 1.1

Create a 4x4 identity matrix using `np.eye`. Then create a 4x4 matrix of random integers between 1 and 20 (inclusive). Print both.

In [ ]:
# YOUR CODE HERE

### Exercise 1.2

Create an array of 100 evenly spaced values between -pi and pi using `np.linspace`.

In [ ]:
# YOUR CODE HERE

---
## 2. Indexing and Slicing

NumPy indexing extends Python list indexing to multiple dimensions. Key concepts: basic slicing returns **views** (not copies), boolean indexing returns **copies**.

In [ ]:
# 1D slicing
x = np.arange(10)
print("x:", x)
print("x[2:5]:", x[2:5])
print("x[::2]:", x[::2])       # every other element
print("x[::-1]:", x[::-1])     # reversed

In [ ]:
# 2D indexing
m = np.arange(12).reshape(3, 4)
print("m:\n", m)
print("row 1:", m[1])            # entire row
print("col 2:", m[:, 2])         # entire column
print("submatrix:\n", m[0:2, 1:3])  # top-left 2x2 block

In [ ]:
# Boolean indexing -- returns a copy
vals = np.array([3, -1, 4, -1, 5, 9])
print("positives:", vals[vals > 0])

# Fancy indexing -- indexing with an array of indices
indices = np.array([0, 2, 4])
print("fancy:", vals[indices])

### Exercise 2.1

Given the matrix below, extract:
1. The last column
2. All elements greater than 5
3. The 2x2 submatrix from the bottom-right corner

In [ ]:
mat = np.arange(1, 17).reshape(4, 4)
print("mat:\n", mat)

# YOUR CODE HERE

### Exercise 2.2

Given an array of exam scores, replace all scores below 60 with 60 (floor them). Do it in one line, no loops.

In [ ]:
scores = np.array([85, 42, 91, 55, 73, 38, 67, 100, 49, 78])

# YOUR CODE HERE

print(scores)  # should show [85, 60, 91, 60, 73, 60, 67, 100, 60, 78]

---
## 3. Broadcasting

Broadcasting lets NumPy operate on arrays of different shapes without copying data. The rules:

1. If arrays differ in number of dimensions, the shape of the smaller one is padded with 1s **on the left**.
2. Arrays with size 1 along a dimension act as if they had the size of the largest array in that dimension.
3. If sizes disagree and neither is 1, you get an error.

Common pattern: subtracting the mean from each column of a matrix.

In [ ]:
# Scalar broadcast
a = np.array([1, 2, 3])
print("a * 10:", a * 10)  # scalar broadcasts to [10, 10, 10]

# Column-wise operation: subtract mean of each column
data = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0],
                 [7.0, 8.0, 9.0]])

col_means = data.mean(axis=0)  # shape (3,)
centered = data - col_means    # (3,3) - (3,) -> broadcasts row-wise
print("means:", col_means)
print("centered:\n", centered)

In [ ]:
# Row-wise operation: divide each row by its sum (need a column vector)
row_sums = data.sum(axis=1, keepdims=True)  # shape (3, 1) -- keepdims is key
normalized = data / row_sums
print("row sums shape:", row_sums.shape)
print("normalized:\n", normalized)
print("row sums of normalized:", normalized.sum(axis=1))  # should be [1, 1, 1]

### Exercise 3.1

Given a matrix of shape (5, 3), z-score normalize each column: `(x - mean) / std`. Do it with broadcasting, no loops.

In [ ]:
rng = np.random.default_rng(0)
X = rng.random((5, 3)) * 100  # random data
print("X:\n", X)

# YOUR CODE HERE
# X_normalized = ...

# Verify: each column should have mean ~ 0 and std ~ 1
# print("means:", X_normalized.mean(axis=0))
# print("stds:", X_normalized.std(axis=0))

### Exercise 3.2

Compute the outer product of two vectors `u = [1, 2, 3]` and `v = [4, 5]` using broadcasting (not `np.outer`). Hint: reshape one of them to a column vector.

In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5])

# YOUR CODE HERE
# Expected result:
# [[ 4,  5],
#  [ 8, 10],
#  [12, 15]]

---
## 4. Vectorized Operations

NumPy operations work element-wise by default. Matrix multiplication uses `@` (or `np.dot`). Universal functions (ufuncs) like `np.exp`, `np.log`, `np.sin` operate element-wise and are implemented in C.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# Element-wise
print("a + b:", a + b)
print("a * b:", a * b)    # NOT matrix multiply
print("a ** 2:", a ** 2)

# Dot product / matrix multiply
print("dot:", a @ b)       # 1D @ 1D = scalar (dot product)

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("A @ B:\n", A @ B)   # 2D @ 2D = matrix multiply

In [ ]:
# Ufuncs
x = np.array([0, np.pi/4, np.pi/2])
print("sin:", np.sin(x))
print("exp:", np.exp(np.array([0, 1, 2])))

# Aggregations
data = np.array([[1, 2], [3, 4], [5, 6]])
print("sum all:", data.sum())
print("sum cols:", data.sum(axis=0))
print("max rows:", data.max(axis=1))

### Exercise 4.1

Implement the softmax function for a 1D array: `softmax(x)_i = exp(x_i) / sum(exp(x))`. Use vectorized ops only. For numerical stability, subtract `max(x)` before exponentiating.

In [ ]:
def softmax(x):
    # YOUR CODE HERE
    pass

logits = np.array([2.0, 1.0, 0.1])
# probs = softmax(logits)
# print("probs:", probs)
# print("sum:", probs.sum())  # should be 1.0


### Exercise 4.2

Given two matrices `A` (3x2) and `B` (2x4), compute their matrix product using `@`. Then compute the element-wise product of `A` with its transpose. What shapes do you get?

In [ ]:
A = np.array([[1, 2], [3, 4], [5, 6]])
B = np.array([[1, 0, 2, 1], [0, 1, 1, 2]])

# YOUR CODE HERE

---
## 5. Reshaping

`reshape` returns a view when possible (no data copy). `flatten` always copies. `ravel` returns a view when possible. `np.expand_dims` adds a dimension of size 1 (useful for broadcasting).

In [ ]:
a = np.arange(12)
print("original:", a.shape)

b = a.reshape(3, 4)
print("reshaped:\n", b)

# -1 infers the dimension
c = a.reshape(2, -1)  # 2 rows, NumPy figures out 6 columns
print("auto shape:", c.shape)

# Transpose
print("transpose shape:", b.T.shape)  # (3,4) -> (4,3)

# Flatten vs ravel
print("flatten:", b.flatten())  # always a copy
print("ravel:", b.ravel())     # view when possible

In [ ]:
# expand_dims -- critical for broadcasting
v = np.array([1, 2, 3])  # shape (3,)
print("original shape:", v.shape)

col = np.expand_dims(v, axis=1)  # shape (3, 1) -- column vector
print("column vector shape:", col.shape)

row = np.expand_dims(v, axis=0)  # shape (1, 3) -- row vector
print("row vector shape:", row.shape)

# Equivalent shorthand: v[:, np.newaxis] or v[:, None]
print("newaxis shape:", v[:, None].shape)

### Exercise 5.1

Take a 1D array of 24 elements. Reshape it into shapes (2, 3, 4), (6, 4), and (4, 6). Print each.

In [ ]:
arr = np.arange(24)

# YOUR CODE HERE

### Exercise 5.2

You have a batch of 8 grayscale images stored as a (8, 28, 28) array. A convolutional layer expects input of shape (batch, channels, height, width). Add a channel dimension at position 1 to get shape (8, 1, 28, 28). Use `np.expand_dims`.

In [ ]:
images = np.zeros((8, 28, 28))  # fake batch of images

# YOUR CODE HERE
# images_4d = ...

# print(images_4d.shape)  # should be (8, 1, 28, 28)

---
## 6. Why Vectorization Matters

Python `for` loops over array elements are slow because each iteration involves Python interpreter overhead (type checking, object creation, dispatch). NumPy vectorized ops run in compiled C, operating on contiguous memory blocks. The difference is typically 10--100x.

This is the single most important NumPy concept for ML. If your training loop is slow, the first thing to check is whether you have Python loops that could be vectorized.

In [ ]:
import time

n = 1_000_000
a = np.random.default_rng(42).random(n)
b = np.random.default_rng(43).random(n)

# Python loop
start = time.perf_counter()
result_loop = 0.0
for i in range(n):
    result_loop += a[i] * b[i]
time_loop = time.perf_counter() - start

# Vectorized
start = time.perf_counter()
result_vec = a @ b
time_vec = time.perf_counter() - start

print(f"Loop:       {time_loop:.4f}s  (result: {result_loop:.6f})")
print(f"Vectorized: {time_vec:.6f}s  (result: {result_vec:.6f})")
print(f"Speedup:    {time_loop / time_vec:.0f}x")

### Exercise 6.1

Compute the Euclidean distance between two vectors (1 million elements each) two ways:
1. With a Python `for` loop
2. With `np.sqrt(np.sum((a - b) ** 2))`

Time both and print the speedup.

In [ ]:
rng = np.random.default_rng(99)
a = rng.random(1_000_000)
b = rng.random(1_000_000)

# YOUR CODE HERE

---
## Self-Assessment

You should be able to answer these without looking anything up:

1. **What is the difference between `a * b` and `a @ b` for 2D arrays?**

2. **You have an array of shape (3, 1) and another of shape (1, 4). What shape results from adding them, and why?**

3. **Why does `a.reshape(3, 4)` usually not copy data, but `a[a > 0]` does?**

4. **Your colleague's code computes a pairwise distance matrix with two nested for loops over N data points. It takes 10 minutes. What would you suggest?**

If you struggled with any of these, review the resources below before the course starts. These concepts come up constantly in ML implementations.

---
## References

- [NumPy Quickstart Tutorial](https://numpy.org/doc/stable/user/quickstart.html) -- official, concise, covers all the essentials.
- [NumPy for MATLAB Users](https://numpy.org/doc/stable/user/numpy-for-matlab-users.html) -- useful comparison if you have a MATLAB background.
- [From Python to NumPy](https://www.labri.fr/perso/nrougier/from-python-to-numpy/) -- Nicolas Rougier's free book; excellent chapter on vectorization techniques.